# 04 · silver/processing ➜ silver  (and ➜ gold/processing)

This is the **Bronze-to-Silver notebook that runs inside the Azure pipeline**. It uses the same cleaning rules we worked out in
`01_bronze_to_silver_local` and the weather logic from `02_weather_api_to_silver`, packed into functions with no exploration charts, so it runs fast and cheaply in Synapse.

### Where this notebook sits in the pipeline
```
bronze/new/*.csv
   │  Data Factory: Get Metadata → ForEach → Copy
   ├──► bronze/archive/<file>/<file>_<timestamp>.csv      (the untouched original, kept forever)
   └──► silver/processing/*.csv                            (a working copy for this run)
                │
                │  Data Factory: Notebook activity → THIS NOTEBOOK
                ▼
        silver/<sensor>/<place>__<timestamp>.parquet       (cleaned, one file per run)
        silver/weather/<place>_weather.parquet             (one growing weather file per place)
        bronze/archive/weather/<place>__<month>.json       (raw API answers)
        gold/processing/<place>_<sensor>__<timestamp>.parquet   (hand-over to notebook 05)
                │
                │  on success: Data Factory deletes silver/processing/*  (ready for the next batch)
                │  on failure: files stay in silver/processing for you to inspect
                ▼
        notebook 05 (gold/processing ➜ gold)
```

### Why "processing" folders?
They work like an **in-tray**. Whatever is in the tray is "today's work". When the job succeeds, the tray is emptied.
When it fails, the files stay put, nothing is lost or half-written, and you can see exactly what broke.
That's why this notebook **stops with an error** if any quality check fails: Data Factory sees the failure and takes the failure path.

### Key design decisions (and why)
| What we do | Why |
|---|---|
| One `RUN_LOCAL` switch | The same code runs on the laptop and in Synapse, so there's nothing to keep in sync |
| Weather date range from the **cleaned UTC** data | The weather matches the hive data to the exact hour |
| Negative weight → clip at 0 | An empty scale can't weigh less than nothing |
| Flow error code 7999 and impossible night spikes → NaN | Shown in notebook 01: physically impossible |
| Weather fetched month by month, cached in bronze | A failure costs one month; re-runs only download what's new |
| Weather **appended** to one weather file per place | The LMS asks to "append to a single weather file" |
| File names validated; unknown files skipped with a message | Resilience to bad data drops (bonus challenge) |
| Quality checks **before** writing; fail loudly | So Data Factory's failure path works |

## 0. Settings: the only cell you change when moving between your laptop and Azure
The notebook **detects by itself** where it runs:
- on your **laptop** it uses the folders of the **Bee Haven- Azure** project (the notebook sits in `notebooks/`),
- in **Synapse** it reads and writes the Azure Data Lake (`abfss://...`). Fill in your container, storage account and key below; the notebook stops with a clear message if you forget.

**First time on your laptop?** These notebooks need `fsspec` (Synapse already has it). Run once in a cell: `%pip install fsspec requests pyarrow`, then restart.

**Never paste a real account key into a notebook you share or upload.** Anyone who sees the key controls the whole data lake.
Better options, from best to OK:
1. In Synapse, read the key from **Azure Key Vault** (commented line below).
2. Put it in the notebook only while testing, and delete it before sharing.

In [ ]:
import re
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import fsspec

# Detect automatically where we run: Synapse has the 'notebookutils' package, a laptop doesn't.
try:
    from notebookutils import mssparkutils
    RUN_LOCAL = False                # running in Synapse / Azure
except ImportError:
    RUN_LOCAL = True                 # running on the laptop

if RUN_LOCAL:
    root_dir = Path('..').resolve().as_posix() + '/'          # the "Bee Haven- Azure" folder
    storage_options = {}
else:
    # abfss://<container>@<storage account>.dfs.core.windows.net/
    root_dir = 'abfss://<YOUR CONTAINER>@<YOUR STORAGE ACCOUNT>.dfs.core.windows.net/'
    storage_options = {
        'account_name': '<YOUR STORAGE ACCOUNT>',
        'account_key': '<YOUR ACCOUNT KEY>',   # Storage account -> Security + networking -> Access keys
        # safer: 'account_key': mssparkutils.credentials.getSecret('<key vault name>', '<secret name>'),
    }
    if '<YOUR' in root_dir or '<YOUR' in str(storage_options):
        raise ValueError('Fill in your container, storage account and key in this cell first.')

# the medallion folders
bronze_new             = root_dir + 'bronze/new/'
bronze_archive_weather = root_dir + 'bronze/archive/weather/'
silver_processing_path = root_dir + 'silver/processing/'
silver_path            = root_dir + 'silver/'
gold_processing        = root_dir + 'gold/processing/'
gold_path              = root_dir + 'gold/'

PLACES = {                                       # hive locations we know about
    'schwartau': {'lat': 53.919444, 'lon': 10.6975},
    'wurzburg':  {'lat': 49.783333, 'lon': 9.933333},
}
SENSORS = ['flow', 'humidity', 'temperature', 'weight']
LOCAL_TZ = 'Europe/Berlin'                       # hive loggers write German clock time

RUN_STAMP = pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%dT%Hh%Mm%Ss')   # one stamp for every file this run writes
print('root:', root_dir)
print('run stamp:', RUN_STAMP)

## 1. Small file helpers
**Why:** reading and writing in Azure needs `fsspec` (the "Azure file system" library), while your laptop uses normal folders.
These helpers hide the difference, so the rest of the notebook is *exactly the same* in both places.
`fsspec` picks the right file system from the path itself (`abfss://...` → Azure, `/Users/...` → local disk).

In [ ]:
FS, _ = fsspec.core.url_to_fs(root_dir, **storage_options)

def list_files(folder, pattern='*'):
    """Files in a folder matching a pattern, ignoring hidden files like .DS_Store."""
    return sorted(p for p in FS.glob(folder + pattern) if not p.rsplit('/', 1)[-1].startswith('.'))

def file_name(path):
    return path.rstrip('/').rsplit('/', 1)[-1]

def _make_folder(path):
    folder = path.rsplit('/', 1)[0]
    try:
        FS.makedirs(folder, exist_ok=True)
    except Exception:
        pass                          # Azure Data Lake creates folders automatically when writing

def read_csv(path, **kwargs):
    with FS.open(path, 'rb') as f:
        return pd.read_csv(f, **kwargs)

def read_parquet(path):
    with FS.open(path, 'rb') as f:
        return pd.read_parquet(f)

def write_parquet(df, path):
    _make_folder(path)
    with FS.open(path, 'wb') as f:
        df.to_parquet(f, index=False)
    print(f'  wrote {path}  ({len(df):,} rows)')

def write_json(data, path):
    _make_folder(path)
    with FS.open(path, 'w') as f:
        json.dump(data, f, indent=2)

def read_json(path):
    with FS.open(path, 'r') as f:
        return json.load(f)

def exists(path):
    return FS.exists(path)

## 2. (Local only) Do Data Factory's job: copy bronze/new → silver/processing
In Azure, the **Copy activity** in your pipeline puts the CSVs into `silver/processing/`. On your laptop there is no Data Factory,
so this cell does that one step for you, but only if the in-tray is empty. In Synapse (`RUN_LOCAL = False`) it does nothing.

In [ ]:
if RUN_LOCAL and not list_files(silver_processing_path, '*.csv'):
    for src in list_files(bronze_new, '*.csv'):
        dst = silver_processing_path + file_name(src)
        _make_folder(dst)
        FS.copy(src, dst)
        print('copied', file_name(src), '→ silver/processing/')

## 3. Find and check the files in silver/processing
File names tell us **what** a file is and **where** it comes from: `flow_schwartau.csv` → sensor `flow`, place `schwartau`.
Anything after the place is ignored, so Data Factory names like `flow_schwartau_2025-02-21_13h43m27s.csv` work too.

**Filename validation (bonus challenge):** a file whose sensor or place we don't recognise (a typo, a photo someone uploaded …)
is **skipped with a message**, so one bad file can't crash the whole run.

In [ ]:
batches = {}          # batches[place][sensor] = raw DataFrame
skipped = []

for path in list_files(silver_processing_path):
    name = file_name(path)
    parts = re.split(r'[_.]', name.lower())
    if not name.lower().endswith('.csv') or len(parts) < 3:
        skipped.append((name, 'not a <sensor>_<place>.csv file')); continue
    sensor, place = parts[0], parts[1]
    if sensor not in SENSORS:
        skipped.append((name, f'unknown sensor "{sensor}"')); continue
    if place not in PLACES:
        skipped.append((name, f'unknown place "{place}"')); continue
    if sensor in batches.get(place, {}):
        skipped.append((name, f'second {sensor} file for {place} in the same batch')); continue
    batches.setdefault(place, {})[sensor] = read_csv(path)
    print(f'read {name:45s} → {sensor:12s} {place:10s} {len(batches[place][sensor]):>10,} rows')

for name, why in skipped:
    print(f'SKIPPED {name}: {why}')
if not batches:
    raise FileNotFoundError(f'No usable CSV files in {silver_processing_path}. Nothing to do.')

## 4. Cleaning functions, one per sensor
These are the decisions from notebook 01, as reusable functions. Short reminder of each rule:

| Rule | Why |
|---|---|
| Lowercase snake_case column names | One naming convention across every table |
| Text → datetime, values → numbers | Needed for sorting, maths and time zones |
| German local time → **UTC** (`'infer'` for 1–5 min data, `True` for 12-hourly data) | Removes the October duplicate hour; matches the weather API |
| Flow file = departures stacked on top of arrivals → split into `flow_out` / `flow_in` | One timestamp = one row |
| Humidity: `abs()` | −100 next to 100 is a sign flip |
| Weight: clip at 0 | An empty scale can't weigh less than nothing |
| Flow: 7999 and arrivals above the busiest departure minute → NaN | Error code and impossible spikes (mostly at night) |
| Missing values stay NaN | Silver keeps facts; it doesn't invent data |

In [ ]:
def standardise_columns(df):
    df = df.copy()
    df.columns = [re.sub(r'_+', '_', re.sub(r'[^0-9a-z]+', '_', str(c).strip().lower())).strip('_') for c in df.columns]
    return df

def to_utc(ts, ambiguous):
    return ts.dt.tz_localize(LOCAL_TZ, ambiguous=ambiguous, nonexistent='shift_forward').dt.tz_convert('UTC')

def basic_types(df, value_col):
    df = standardise_columns(df)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    return df.dropna(subset=['timestamp'])          # a row without a time can't be used anywhere


def clean_flow(flow):
    flow = basic_types(flow, 'flow')
    half = len(flow) // 2
    departures = flow.iloc[:half].reset_index(drop=True)
    arrivals = flow.iloc[half:].reset_index(drop=True)
    # prove the "two stacked halves" structure before relying on it
    assert len(flow) % 2 == 0, 'flow file has an odd number of rows'
    assert (departures['timestamp'].values == arrivals['timestamp'].values).all(), 'flow halves have different timestamps'
    assert (departures['flow'].dropna() <= 0).all() and (arrivals['flow'].dropna() >= 0).all(), 'flow halves have unexpected signs'

    departures['timestamp'] = to_utc(departures['timestamp'], 'infer')
    arrivals['timestamp'] = to_utc(arrivals['timestamp'], 'infer')
    out = (departures.rename(columns={'flow': 'flow_out'})
           .merge(arrivals.rename(columns={'flow': 'flow_in'}), on='timestamp', validate='one_to_one'))

    max_departure = out['flow_out'].abs().max()
    bad = (out['flow_in'] == 7999) | (out['flow_in'] > max_departure)
    out.loc[bad, 'flow_in'] = np.nan
    out[['flow_out', 'flow_in']] = out[['flow_out', 'flow_in']].astype('Int64')
    print(f'  flow: {bad.sum()} impossible arrival values set to NaN')
    return out


def clean_humidity(humidity):
    humidity = basic_types(humidity, 'humidity')
    humidity['timestamp'] = to_utc(humidity['timestamp'], True)
    print(f"  humidity: {(humidity['humidity'] < 0).sum()} sign-flipped values fixed")
    humidity['humidity'] = humidity['humidity'].abs()
    humidity.loc[humidity['humidity'] > 100, 'humidity'] = np.nan
    return humidity


def clean_temperature(temperature):
    temperature = basic_types(temperature, 'temperature')
    temperature['timestamp'] = to_utc(temperature['timestamp'], 'infer')
    temperature.loc[~temperature['temperature'].between(-30, 60), 'temperature'] = np.nan
    return temperature


def clean_weight(weight):
    weight = basic_types(weight, 'weight')
    weight['timestamp'] = to_utc(weight['timestamp'], True)
    print(f"  weight: {(weight['weight'] < 0).sum()} values below 0 set to 0")
    weight['weight'] = weight['weight'].clip(lower=0)
    return weight


CLEANERS = {'flow': clean_flow, 'humidity': clean_humidity, 'temperature': clean_temperature, 'weight': clean_weight}

def finish(df):
    """Last steps for every table: no duplicate rows, sorted by time."""
    return df.drop_duplicates().drop_duplicates(subset='timestamp', keep='last').sort_values('timestamp').reset_index(drop=True)

## 5. Quality checks
**Why:** "the code ran" isn't the same as "the data is right". These checks run **before** anything is written.
If one fails, the notebook raises an error, Data Factory marks the run as failed, and the files stay in `silver/processing`.

In [ ]:
def check(df, sensor):
    problems = []
    if str(df['timestamp'].dt.tz) != 'UTC':           problems.append('timestamps not UTC')
    if not df['timestamp'].is_unique:                  problems.append('duplicate timestamps')
    if not df['timestamp'].is_monotonic_increasing:    problems.append('timestamps not sorted')
    if sensor == 'flow':
        if (df['flow_out'].dropna() > 0).any() or (df['flow_in'].dropna() < 0).any(): problems.append('flow signs wrong')
    if sensor == 'humidity' and not df['humidity'].dropna().between(0, 100).all():  problems.append('humidity outside 0-100')
    if sensor == 'weight' and (df['weight'].dropna() < 0).any():                     problems.append('negative weight')
    if problems:
        raise ValueError(f'{sensor}: ' + '; '.join(problems))

## 6. Weather functions (BrightSky API)
Same logic as notebook 02:
1. **Fetch** month by month. Each raw answer is saved in `bronze/archive/weather/`, and months already saved are **not downloaded again**.
2. **Flatten** the JSON: one row per hour, plus `<measure>_source_distance` from `fallback_source_ids` (how far away each value was measured).
3. **Clean**: UTC, duplicates, numbers, the `condition` category, plausibility ranges.
4. **Append** to one weather file per place in silver (`silver/weather/<place>_weather.parquet`), replacing hours we already had.

In [ ]:
import requests

URL = 'https://api.brightsky.dev/weather'
HEADERS = {'Accept': 'application/json'}
NOT_MEASURES = {'timestamp', 'source_id', 'fallback_source_ids', 'condition', 'icon',
                'precipitation_probability', 'precipitation_probability_6h'}
DROP_COLUMNS = ['source_id', 'fallback_source_ids', 'icon', 'visibility',
                'precipitation_probability', 'precipitation_probability_6h']
CONDITIONS = ['dry', 'fog', 'rain', 'sleet', 'snow', 'hail', 'thunderstorm']
PLAUSIBLE = {'temperature': (-30, 40), 'dew_point': (-40, 30), 'relative_humidity': (0, 100),
             'cloud_cover': (0, 100), 'precipitation': (0, 100), 'pressure_msl': (940, 1060),
             'wind_speed': (0, 200), 'wind_gust_speed': (0, 250), 'wind_direction': (0, 360),
             'wind_gust_direction': (0, 360), 'sunshine': (0, 60), 'solar': (0, 2)}


def fetch_period(place, start, end, retries=3):
    params = PLACES[place] | {'date': start.isoformat(), 'last_date': end.isoformat()}
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(URL, headers=HEADERS, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            print(f'    attempt {attempt}: status {r.status_code}: {r.text[:120]}')
        except requests.RequestException as e:
            print(f'    attempt {attempt}: {type(e).__name__}: {e}')
        time.sleep(2 * attempt)
    raise RuntimeError(f'Could not fetch weather for {place} {start} → {end}')


def month_chunks(start, end):
    edges = pd.date_range(start.floor('D'), end.ceil('D'), freq='MS', tz='UTC')
    edges = sorted({start, end, *[e for e in edges if start < e < end]})
    return list(zip(edges[:-1], edges[1:]))


def flatten(weather_json):
    distance = {s['id']: s['distance'] for s in weather_json['sources']}
    rows = []
    for record in weather_json['weather']:
        row = dict(record)
        fallback = record.get('fallback_source_ids') or {}
        for measure in record:
            if measure not in NOT_MEASURES:
                row[f'{measure}_source_distance'] = distance.get(fallback.get(measure, record['source_id']))
        rows.append(row)
    return pd.DataFrame(rows).drop(columns=DROP_COLUMNS, errors='ignore')


def clean_weather(df):
    df = standardise_columns(df)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    measure_cols = [c for c in df.columns if c not in ('timestamp', 'condition')]
    df[measure_cols] = df[measure_cols].apply(pd.to_numeric, errors='coerce').astype('float64')
    for col, (lo, hi) in PLAUSIBLE.items():
        if col in df:
            df.loc[df[col].notna() & ~df[col].between(lo, hi), col] = np.nan
    if 'condition' in df:
        df['condition'] = df['condition'].astype('string').str.strip().str.lower()
        df['condition'] = pd.Categorical(df['condition'], categories=CONDITIONS)
    return finish(df)


def get_weather(place, start, end):
    """Raw JSON → bronze (cached), cleaned → appended to silver. Returns this batch's weather."""
    start, end = start.floor('60min'), end.ceil('60min')
    frames = []
    for a, b in month_chunks(start, end):
        path = f"{bronze_archive_weather}{place}__{a:%Y-%m-%dT%H}_{b:%Y-%m-%dT%H}.json"
        if exists(path):
            data = read_json(path)
        else:
            data = fetch_period(place, a, b)
            write_json(data, path)
            time.sleep(0.5)                     # be polite to a free API
        frames.append(flatten(data))
    batch = clean_weather(pd.concat(frames, ignore_index=True))
    batch = batch[batch['timestamp'].between(start, end)].reset_index(drop=True)
    print(f'  weather: {len(batch):,} hours from {len(frames)} monthly files')

    # append to the single silver weather file for this place
    silver_weather = f'{silver_path}weather/{place}_weather.parquet'
    if exists(silver_weather):
        old = read_parquet(silver_weather)
        old = old[~old['timestamp'].isin(batch['timestamp'])]        # newer download wins
        combined = finish(pd.concat([old, batch], ignore_index=True))
    else:
        combined = batch
    if 'condition' in combined:
        combined['condition'] = pd.Categorical(combined['condition'].astype('string'), categories=CONDITIONS)
    write_parquet(combined, silver_weather)
    return batch

## 7. Run it: clean, check, write, for every place found
For each place, and each sensor in this batch:
1. clean → 2. check → 3. write to **silver** (`silver/<sensor>/<place>__<run stamp>.parquet`, a permanent cleaned copy of this batch)
→ 4. write to **gold/processing** (`<place>_<sensor>__<run stamp>.parquet`, the hand-over to notebook 05).

Then the weather for exactly the period this batch covers.

Every file of one run gets the **same run stamp**, so you can always tell which files belong together.

In [ ]:
summary = []
for place, sensors in batches.items():
    print(f'\n===== {place} =====')
    cleaned = {}
    for sensor, raw_df in sensors.items():
        df = finish(CLEANERS[sensor](raw_df))
        check(df, sensor)
        cleaned[sensor] = df
        summary.append({'place': place, 'table': sensor, 'rows': len(df),
                        'from': df['timestamp'].min(), 'to': df['timestamp'].max()})

    # only write once ALL sensors of this place passed their checks
    for sensor, df in cleaned.items():
        write_parquet(df, f'{silver_path}{sensor}/{place}__{RUN_STAMP}.parquet')
        write_parquet(df, f'{gold_processing}{place}_{sensor}__{RUN_STAMP}.parquet')

    start = min(df['timestamp'].min() for df in cleaned.values())
    end = max(df['timestamp'].max() for df in cleaned.values())
    weather = get_weather(place, start, end)
    write_parquet(weather, f'{gold_processing}{place}_weather__{RUN_STAMP}.parquet')
    summary.append({'place': place, 'table': 'weather', 'rows': len(weather),
                    'from': weather['timestamp'].min(), 'to': weather['timestamp'].max()})

pd.DataFrame(summary)

## 8. (Local only) Empty the in-tray, as Data Factory does on success
In Azure, add a **Delete activity** after the Notebook activity on the **green (success)** arrow, pointing at `silver/processing/`.
On the **red (failure)** arrow you can, for example, copy the files to `silver/failed/<run time>/` so you can inspect them later.
On your laptop this cell plays the Delete activity's part.

In [ ]:
if RUN_LOCAL:
    for path in list_files(silver_processing_path, '*.csv'):
        FS.rm(path)
    print('silver/processing emptied')

## Release the Spark pool (Synapse only)
**Why:** in Synapse you pay for the Spark pool while the session is open. Stopping it as soon as the work is done saves your Azure credits.

In [ ]:
if not RUN_LOCAL:
    mssparkutils.session.stop()